In [ ]:
from IPython.display import Audio, display
import matplotlib.pyplot as plt
import librosa
display(Audio("./VisualEchoes/data/sweep_audio/3ms_sweep.wav", rate=44100))


In [ ]:
import matplotlib.pyplot as plt
import librosa

chirp = librosa.load("./VisualEchoes/data/sweep_audio/3ms_sweep.wav", sr=44100)[0]
plt.plot(chirp)
plt.title("Chirp Signal")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.grid()
plt.show()

In [ ]:
import os
# --- KONFIGURACJA ŚCIEŻEK ---
scene = "room_0"
data_path = "./habitat-sim/data/scene_datasets/replica"
metadata_path = f"./habitat-sim/metadata/replica/{scene}/graph.pkl"
output_base_dir = f"./habitat-sim/data/echoes-replica/{scene}/3ms_sweep"
navmesh_path = os.path.join(data_path, scene, "habitat/mesh_semantic.navmesh")


In [ ]:
import os
import os.path as osp
import sys


# To musi być przed jakimkolwiek importem habitat_sim
import habitat_sim
print("Import zakończony sukcesem!")

In [ ]:
import os
import sys
import habitat_sim
import soundspaces
import pickle
import numpy as np
import soundfile as sf
from habitat_sim.utils.common import quat_from_angle_axis
import matplotlib.pyplot as plt

# 1. WERYFIKACJA ŚCIEŻEK
scene_config = os.path.join(data_path, scene, "habitat/replica_stage.stage_config.json")
navmesh_path = os.path.join(data_path, scene, "habitat/mesh_semantic.navmesh")
if not os.path.exists(scene_config):
    raise FileNotFoundError(f"Brak pliku konfiguracji sceny: {scene_config}")


os.environ['GALLIUM_DRIVER'] = 'llvmpipe'
os.environ['MESA_GL_VERSION_OVERRIDE'] = '4.1'

# 2. KONFIGURACJA SYMULATORA
cfg = habitat_sim.SimulatorConfiguration()
cfg.scene_id = scene_config
cfg.create_renderer = True   # Dla SoundSpaces zaleca się True dla stabilności backendu
cfg.enable_physics = False
cfg.gpu_device_id = 0        # Kluczowe dla WSL

# # --- DEFINICJA SENSORÓW ---


rgb_sensor_spec = habitat_sim.SensorSpec()
rgb_sensor_spec.uuid = "color_sensor"
rgb_sensor_spec.sensor_type = habitat_sim.SensorType.COLOR
rgb_sensor_spec.resolution = np.array([128, 128], dtype=np.int32)
rgb_sensor_spec.orientation = np.array([0, 0, 0]) # Brak obrotu (facing forward)
rgb_sensor_spec.position = np.array([0, 1.5, 0]) # Standardowa wysokość kamery
# 2. Specyfikacja sensora akustycznego (Klucz do Twoich ech)
# SoundSpaces v0.1.7 używa specjalnego typu sensora, który rejestruje echa

# KLUCZOWE: Musisz wskazać SoundSpaces, gdzie są dane RIR i metadane
# Załóżmy, że Twoje dane są w folderze 'data'
acoustic_sensor_spec = habitat_sim.SensorSpec()
acoustic_sensor_spec.uuid = "acoustics_sensor" # To musi pasować do Twojego kodu
acoustic_sensor_spec.sensor_type = habitat_sim.SensorType.NONE # W v0.1.7 SoundSpaces podpina się pod ten typ
# Dodatkowe parametry dla SoundSpaces są zazwyczaj brane z konfiguracji Simulator (cfg)
# Ale sensor musi być na liście specyfikacji agenta.

# 3. KONFIGURACJA AGENTA
agent_cfg = habitat_sim.agent.AgentConfiguration()

agent_cfg.sensor_specifications = [rgb_sensor_spec, acoustic_sensor_spec] 

# 4. INICJALIZACJA
try:
    if 'sim' in locals(): sim.close()
    sim = habitat_sim.Simulator(habitat_sim.Configuration(cfg, [agent_cfg]))
    
    if os.path.exists(navmesh_path):
        sim.pathfinder.load_nav_mesh(navmesh_path)
        print("Navmesh loaded: True")
        
    print("SYSTEM HABITAT GOTOWY")
except Exception as e:
    print("Błąd podczas inicjalizacji:", e)

# 5. WCZYTANIE GRAFU (Źródło prawdy o pozycjach XYZ)
if "graph" not in globals():
    with open(metadata_path, "rb") as f:
        graph = pickle.load(f)

# Pobranie instancji agenta (standard w v0.1.7)
agent = sim.get_agent(0)

# 6. GŁÓWNA PĘTLA GENERUJĄCA (Gęste próbkowanie 2.0)
for node_id in graph.nodes():
    pos = graph.nodes[node_id]['point'] # Fizyczne XYZ z metadanych [2, 3]
    
    for angle in range(0, 360, 10): # Twoje ulepszenie: krok 10 stopni [4, 5]
        agent_state = habitat_sim.AgentState()
        agent_state.position = pos
        # print(f"Node {node_id}: Pozycja {pos}, Kąt {angle}°")
        # POPRAWKA: Obrót o angle_rad wokół osi Y (wektor [6])
        # W Twoim kodzie był błąd np.array([1]), co powodowało segfault lub błędną rotację
        angle_rad = np.deg2rad(angle)
        agent_state.rotation = quat_from_angle_axis(angle_rad, np.array([0.0, 1.0, 0.0]))
        agent.set_state(agent_state)
        
        # 7. GENEROWANIE ECHA
        obs = sim.get_sensor_observations()
        echo_data = obs['acoustics_sensor']  # To jest klucz do Twojego sensora akustycznego (może być inny w zależności od konfiguracji SoundSpaces)
        img = echo_data  # (84, 84, 4)

        plt.imshow(img.astype("uint8"))
        plt.title("Echo tensor (84x84x4)")
        plt.axis("off")
        plt.show()
        print(f"DEBUG: Kształt danych echa: {echo_data.shape}")

        if len(echo_data.shape) == 3:
            print("BŁĄD: Otrzymano OBRAZ zamiast dźwięku!")
        # To potwierdzi, że Habitat renderuje widok wizualny zamiast audio
        # Nazwa sensora musi być zgodna z Twoim setupem SoundSpaces (np. 'rgb_static_acoustics_sensor')
        sensor_name = 'acoustics_sensor'
        if sensor_name in obs:
            echo_signal = obs[sensor_name]
            
            # 8. ZAPIS DO PLIKU .WAV
            # Używamy częstotliwości 44.1 kHz dla datasetu Replica [7, 8]
            output_path = f"{output_base_dir}/{angle}/{node_id}.wav"
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            sf.write(output_path, echo_signal, 44100)
        else:
            # Jeśli sensor nie jest zainicjowany, echo_signal nie powstanie
            pass

In [ ]:
import habitat_sim
from habitat_sim.utils.common import quat_from_angle_axis
import numpy as np
import pickle
import os
from PIL import Image
# 1. Konfiguracja i inicjalizacja (zakładam, że 'sim' jest już gotowy)
scene = "apartment_0"
metadata_path = f"./habitat-sim/metadata/replica/{scene}/graph.pkl"
output_gif_path = f"view_360_{scene}.gif"

# 2. Pobranie pierwszej lokalizacji z grafu (Twoje źródło prawdy XYZ)
with open(metadata_path, "rb") as f:
    graph = pickle.load(f)

# Pobieramy ID pierwszego węzła i jego fizyczną pozycję [1, 2]
all_nodes = list(graph.nodes())
first_node_id = all_nodes[0] if all_nodes else None
print(f"Pierwszy węzeł ID: {first_node_id}")
target_position = graph.nodes[first_node_id]['point']

print(f"Generowanie GIF dla lokalizacji ID: {first_node_id}")
print(f"Współrzędne XYZ: {target_position}")

# 3. Przygotowanie agenta
agent = sim.get_agent(0)
frames = []

# 4. Pętla obrotu co 10 stopni (Visual Echoes 2.0) [3, 4]
for angle in range(0, 360, 10):
    agent_state = habitat_sim.AgentState()
    agent_state.position = target_position
    
    # Obrót wokół osi Y (pionowej) [5, 6]
    angle_rad = np.deg2rad(angle)
    agent_state.rotation = quat_from_angle_axis(angle_rad, np.array([0, 1.0, 0]))
    agent.set_state(agent_state)
    
    # Pobranie obserwacji wizualnej
    obs = sim.get_sensor_observations()
    print(obs.keys())  # Sprawdź dostępne sensory, aby upewnić się, że 'color_sensor' jest obecny
    rgb_image = obs["rgb_sensor"] # upewnij się, że nazwa w cfg to 'color_sensor'
    
    # Konwersja na format PIL Image i dodanie do listy
    frames.append(Image.fromarray(rgb_image))

# 5. Generowanie i zapis GIF
if frames:
    frames.save(
        output_gif_path,
        save_all=True,
        append_images=frames[1:],
        duration=100, # czas trwania klatki w ms
        loop=0        # 0 oznacza zapętlenie w nieskończoność
    )
    print(f"SUKCES: GIF został zapisany jako {output_gif_path}")
else:
    print("Błąd: Nie wygenerowano żadnych klatek.")